##### Imports and function

In [6]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable
from datetime import datetime

CURRENT_TS = datetime.utcnow().strftime("%Y-%m-%dT%H:%M:%S")
HIGH_DATE  = "9999-12-31"

def upsert_silver_scd2(table_name: str, parsed_df, business_key: str = "resource_id"):
    """
    SCD Type 2 upsert into silver schema.
    - New records     → inserted as current
    - Changed records → old row closed, new row inserted
    - Unchanged       → skipped
    """
    incoming = parsed_df \
        .withColumn("effective_from", F.lit(CURRENT_TS)) \
        .withColumn("effective_to",   F.lit(HIGH_DATE)) \
        .withColumn("is_current",     F.lit(True)) \
        .withColumn("row_hash", F.md5(
            F.concat_ws("|", *[c for c in parsed_df.columns if c != business_key])
        ))

    try:
        spark.table(table_name)
        table_exists = True
    except:
        table_exists = False

    if table_exists:
        silver_dt = DeltaTable.forName(spark, table_name)

        # Step 1 — close out changed rows
        silver_dt.alias("old").merge(
            incoming.alias("new"),
            f"old.{business_key} = new.{business_key} \
              AND old.is_current = true \
              AND old.row_hash != new.row_hash"
        ).whenMatchedUpdate(set={
            "is_current":   F.lit(False),
            "effective_to": F.lit(CURRENT_TS)
        }).execute()

        # Step 2 — insert new and changed rows only
        existing_current = spark.table(table_name) \
            .filter("is_current = true") \
            .select(business_key, "row_hash")

        new_rows = incoming.join(
            existing_current,
            on=[business_key, "row_hash"],
            how="left_anti"
        )
        new_rows.write.format("delta") \
            .mode("append") \
            .saveAsTable(table_name)
    else:
        # First run — create fresh
        incoming.write.format("delta") \
            .mode("overwrite") \
            .saveAsTable(table_name)

    current = spark.table(table_name).filter("is_current = true").count()
    total   = spark.table(table_name).count()
    print(f"  {table_name}: {current} current rows | {total} total rows")

print(f"SCD2 function ready | run timestamp: {CURRENT_TS}")

StatementMeta(, 1d621e81-b17a-4199-8c6c-f1d094241a09, 8, Finished, Available, Finished, False)

SCD2 function ready | run timestamp: 2026-06-16T16:38:26


##### Silver Patient

In [7]:
print("=== silver.patient ===")

bronze = spark.table("bronze.patient")

parsed = bronze \
    .withColumn("p", F.from_json(F.col("raw_json"), MapType(StringType(), StringType()))) \
    .select(
        F.col("resource_id"),
        F.col("p.gender").alias("gender"),
        F.col("p.birthDate").alias("birth_date"),
        F.col("p.deceasedBoolean").alias("deceased"),
        F.col("extraction_timestamp"),
        F.col("load_date")
    ) \
    .dropDuplicates(["resource_id"])

upsert_silver_scd2("silver.patient", parsed)

StatementMeta(, 1d621e81-b17a-4199-8c6c-f1d094241a09, 9, Finished, Available, Finished, False)

=== silver.patient ===
  silver.patient: 634 current rows | 634 total rows


##### Silver Encounter

In [8]:
print("=== silver.encounter ===")

bronze = spark.table("bronze.encounter")

parsed = bronze \
    .withColumn("e", F.from_json(F.col("raw_json"), MapType(StringType(), StringType()))) \
    .select(
        F.col("resource_id"),
        F.col("e.status").alias("status"),
        F.col("e.class").alias("encounter_class"),
        F.col("extraction_timestamp"),
        F.col("load_date")
    ) \
    .dropDuplicates(["resource_id"])

upsert_silver_scd2("silver.encounter", parsed)

StatementMeta(, 1d621e81-b17a-4199-8c6c-f1d094241a09, 10, Finished, Available, Finished, False)

=== silver.encounter ===
  silver.encounter: 272 current rows | 272 total rows


##### Silver Observation

In [9]:
print("=== silver.observation ===")

bronze = spark.table("bronze.observation")

parsed = bronze \
    .withColumn("o", F.from_json(F.col("raw_json"), MapType(StringType(), StringType()))) \
    .select(
        F.col("resource_id"),
        F.col("o.status").alias("status"),
        F.col("o.effectiveDateTime").alias("effective_date"),
        F.col("o.subject").alias("subject_ref"),
        F.col("extraction_timestamp"),
        F.col("load_date")
    ) \
    .dropDuplicates(["resource_id"])

upsert_silver_scd2("silver.observation", parsed)

StatementMeta(, 1d621e81-b17a-4199-8c6c-f1d094241a09, 11, Finished, Available, Finished, False)

=== silver.observation ===
  silver.observation: 750 current rows | 750 total rows


##### Silver Condition

In [10]:
print("=== silver.condition ===")

bronze = spark.table("bronze.condition")

parsed = bronze \
    .withColumn("c", F.from_json(F.col("raw_json"), MapType(StringType(), StringType()))) \
    .select(
        F.col("resource_id"),
        F.col("c.clinicalStatus").alias("clinical_status"),
        F.col("c.verificationStatus").alias("verification_status"),
        F.col("c.recordedDate").alias("recorded_date"),
        F.col("c.subject").alias("subject_ref"),
        F.col("extraction_timestamp"),
        F.col("load_date")
    ) \
    .dropDuplicates(["resource_id"])

upsert_silver_scd2("silver.condition", parsed)

StatementMeta(, 1d621e81-b17a-4199-8c6c-f1d094241a09, 12, Finished, Available, Finished, False)

=== silver.condition ===
  silver.condition: 679 current rows | 679 total rows


In [11]:
print("=== Silver verification ===\n")

for resource in ["patient", "encounter", "observation", "condition"]:
    table = f"silver.{resource}"
    current = spark.table(table).filter("is_current = true").count()
    total   = spark.table(table).count()
    print(f"  {table}: {current} current | {total} total")

print("\nSample from silver.patient:")
spark.table("silver.patient").select(
    "resource_id", "gender", "birth_date",
    "effective_from", "effective_to", "is_current", "row_hash"
).show(3)

StatementMeta(, 1d621e81-b17a-4199-8c6c-f1d094241a09, 13, Finished, Available, Finished, False)

=== Silver verification ===

  silver.patient: 634 current | 634 total
  silver.encounter: 272 current | 272 total
  silver.observation: 750 current | 750 total
  silver.condition: 679 current | 679 total

Sample from silver.patient:
+--------------------+------+----------+-------------------+------------+----------+--------------------+
|         resource_id|gender|birth_date|     effective_from|effective_to|is_current|            row_hash|
+--------------------+------+----------+-------------------+------------+----------+--------------------+
|06f47b6b-8d93-40d...|  NULL|      NULL|2026-06-16T16:38:26|  9999-12-31|      true|966d2973d573a124c...|
|               1234W|female|1989-12-07|2026-06-16T16:38:26|  9999-12-31|      true|02323724f55cd8037...|
|           132080109|  male|1995-01-01|2026-06-16T16:38:26|  9999-12-31|      true|4f74ede2e476af601...|
+--------------------+------+----------+-------------------+------------+----------+--------------------+
only showing top 3 rows
